## Score trees based on monophyleticity

> Leaves are named with group code, followed by an underscore.
> We check every internal node, see if all its leaves belong to the same group.
> If yes, such internode is monophyletic.
> The script reports the number of monophyletic internal nodes, the number of all internal nodes, and the fraction of monophyletic internal nodes.

In [1]:
from Bio import Phylo
import re


def score_monophyly(filename):
    """
    Calculate the fraction of internal nodes whose descendant leaves all
    belong to the same group.

    Leaf names must follow this format:
        ABCD_[unique_name]

    Parameters
    ----------
    filename : str
        Path to the input Newick file.

    Returns
    -------
    tuple
        (monophyletic_internal_nodes, all_internal_nodes, fraction)
    """
    tree = Phylo.read(filename, "newick")
    group_pattern = re.compile(r"^([A-Za-z0-9]+)_")

    def get_group(leaf):
        if not leaf.name:
            raise ValueError("Encountered a leaf without a name.")

        match = group_pattern.match(leaf.name)
        if not match:
            raise ValueError(
                f"Invalid leaf name {leaf.name!r}: expected a four-letter "
                "group code followed by an underscore."
            )

        return match.group(1)

    internal_nodes = tree.get_nonterminals()
    monophyletic_nodes = 0

    for node in internal_nodes:
        descendant_groups = {
            get_group(leaf) for leaf in node.get_terminals()
        }

        if len(descendant_groups) == 1:
            monophyletic_nodes += 1

    total_nodes = len(internal_nodes)
    fraction = monophyletic_nodes / total_nodes if total_nodes else 0.0

    #print("-" * 40)
    #print(f"Filename: {filename}")
    print(f"Monophyletic internal nodes: {monophyletic_nodes}")
    print(f"All internal nodes: {total_nodes}")
    print(f"Fraction of monophyletic internal nodes: {fraction:.6f}")
    print("-" * 40)

    # return monophyletic_nodes, total_nodes, fraction

### Flaviviridae NS5 (RdRp)

In [2]:
print("Mifsud")
score_monophyly("nwk/NS5_aa.nwk")
print("ProstT5")
score_monophyly("nwk/NS5_prostt5.nwk")
print("ESM3Di")
score_monophyly("nwk/NS5_esm3di.nwk")


Mifsud
Monophyletic internal nodes: 437
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.950000
----------------------------------------
ProstT5
Monophyletic internal nodes: 434
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.943478
----------------------------------------
ESM3Di
Monophyletic internal nodes: 435
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.945652
----------------------------------------


### Flaviviridae E/E1/E2 mixture

In [3]:
print("AlphaFold - FoldTree")
score_monophyly("nwk/E_af2.nwk")
print("ProstT5 - FoldTree")
score_monophyly("nwk/E_prostt5.nwk")
print("ESM3Di - FoldTree")
score_monophyly("nwk/E_esm3di.nwk")
print("AA - MAFFT - IQ-TREE ModelFinder")
score_monophyly("nwk/E_aa.nwk")

AlphaFold - FoldTree
Monophyletic internal nodes: 490
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.784000
----------------------------------------
ProstT5 - FoldTree
Monophyletic internal nodes: 521
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.833600
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 556
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.889600
----------------------------------------
AA - MAFFT - IQ-TREE ModelFinder
Monophyletic internal nodes: 541
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.865600
----------------------------------------


---